In [1]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version as package_version
import json
import os
from pathlib import Path
import platform
import random
from typing import Any

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from tokenizers import Tokenizer
from tokenizers.decoders import WordPiece as WordPieceDecoder
from tokenizers.models import WordPiece
from tokenizers.normalizers import BertNormalizer
from tokenizers.pre_tokenizers import BertPreTokenizer
from tokenizers.processors import TemplateProcessing
from tokenizers.trainers import WordPieceTrainer

from transformers import PreTrainedTokenizerFast


In [2]:
@dataclass(frozen=True)
class WordPiecePaths:
    """Paths used by the QQP WordPiece-tokenizer pipeline."""

    train_csv_path: Path = Path("data/raw/train.csv")
    processed_dir: Path = Path("data/processed")
    tokenizer_dir: Path = Path(
        "artifacts/tokenizers/wordpiece_uncased_30k"
    )

    @property
    def corpus_path(self) -> Path:
        return self.processed_dir / "tokenizer_corpus.txt"

    @property
    def train_split_path(self) -> Path:
        return self.processed_dir / "train_split.csv"

    @property
    def valid_split_path(self) -> Path:
        return self.processed_dir / "valid_split.csv"

    @property
    def tokenizer_json_path(self) -> Path:
        return self.tokenizer_dir / "tokenizer.json"

    @property
    def tokenizer_config_path(self) -> Path:
        return self.tokenizer_dir / "tokenizer_config.json"

    @property
    def special_tokens_map_path(self) -> Path:
        return self.tokenizer_dir / "special_tokens_map.json"

    @property
    def vocab_path(self) -> Path:
        return self.tokenizer_dir / "vocab.txt"

    @property
    def training_metadata_path(self) -> Path:
        return self.tokenizer_dir / "training_metadata.json"

    def make_dirs(self) -> None:
        """Create all output directories."""
        self.processed_dir.mkdir(parents=True, exist_ok=True)
        self.tokenizer_dir.mkdir(parents=True, exist_ok=True)
        print(">>> Directories created...")

    def validate_input(self) -> None:
        """Check that the original QQP training CSV exists."""
        if not self.train_csv_path.is_file():
            raise FileNotFoundError(
                f">>> QQP training CSV was not found: {self.train_csv_path}"
            )


In [3]:
@dataclass(frozen=True)
class WordPieceConfig:
    """Configuration for training the QQP WordPiece tokenizer."""

    # WordPiece vocabulary
    vocab_size: int = 30_000
    min_frequency: int = 2
    continuing_subword_prefix: str = "##"
    max_input_chars_per_word: int = 100

    # Normalization
    lowercase: bool = True
    strip_accents: bool = True
    clean_text: bool = True
    handle_chinese_chars: bool = True

    # LSTM-Attention and ESIM:
    # each question is encoded separately
    max_sequence_length: int = 64

    # Custom Transformer:
    # both questions are encoded together
    max_pair_length: int = 128

    # Special tokens
    pad_token: str = "[PAD]"
    unk_token: str = "[UNK]"
    cls_token: str = "[CLS]"
    sep_token: str = "[SEP]"
    mask_token: str = "[MASK]"

    # Data split
    valid_size: float = 0.10
    seed: int = 28

    # Used during length analysis
    length_analysis_batch_size: int = 4096

    @property
    def special_tokens(self) -> list[str]:
        """Return special tokens in deterministic order."""
        return [
            self.pad_token,
            self.unk_token,
            self.cls_token,
            self.sep_token,
            self.mask_token,
        ]

In [4]:
cfg = WordPieceConfig()
paths = WordPiecePaths()

paths.make_dirs()
paths.validate_input()

>>> Directories created...


In [5]:
def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    random.seed(seed)

    print(f">>> Seed set to {seed}...")

def save_json(data: Dict[str, Any], path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f">>> JSON saved to {path}...")

def load_json(path: str | Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"JSON file loaded from {path}...")
    return data


seed_everything(cfg.seed)

>>> Seed set to 28...


In [6]:
def load_qqp_data(
    csv_path: str | Path,
) -> pd.DataFrame:
    """Load QQP and remove invalid rows."""

    csv_path = Path(csv_path)

    df = pd.read_csv(csv_path)

    required_columns = [
        "question1",
        "question2",
        "is_duplicate",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required QQP columns: {missing_columns}"
        )

    original_rows = len(df)

    # Keep only the required columns.
    df = df[required_columns].copy()

    # Invalid labels become NaN.
    df["is_duplicate"] = pd.to_numeric(
        df["is_duplicate"],
        errors="coerce",
    )

    # Both questions and the label are required.
    df = df.dropna(
        subset=[
            "question1",
            "question2",
            "is_duplicate",
        ]
    ).copy()

    # Remove surrounding and repeated whitespace.
    for column in ["question1", "question2"]:
        df[column] = (
            df[column]
            .astype(str)
            .str.replace(
                r"\s+",
                " ",
                regex=True,
            )
            .str.strip()
        )

    valid_questions = (
        df["question1"].ne("")
        & df["question2"].ne("")
    )

    valid_labels = df["is_duplicate"].isin(
        [0, 1]
    )

    df = df.loc[
        valid_questions & valid_labels,
        required_columns,
    ].copy()

    df["is_duplicate"] = (
        df["is_duplicate"].astype(np.int8)
    )

    df = df.reset_index(drop=True)

    removed_rows = original_rows - len(df)

    print(
        f">>> QQP loaded from: {csv_path}\n"
        f">>> Original rows: {original_rows:,}\n"
        f">>> Valid rows: {len(df):,}\n"
        f">>> Removed invalid rows: {removed_rows:,}"
    )

    return df

In [7]:
df = load_qqp_data(paths.train_csv_path)
df.head()

>>> QQP loaded from: data\raw\train.csv
>>> Original rows: 404,290
>>> Valid rows: 404,287
>>> Removed invalid rows: 3


,question1,question2,is_duplicate
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [8]:
print(df["is_duplicate"].value_counts(
    normalize=True
), "\n")
df.info()

is_duplicate
0    0.630799
1    0.369201
Name: proportion, dtype: float64 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404287 entries, 0 to 404286
Data columns (total 3 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   question1     404287 non-null  object
 1   question2     404287 non-null  object
 2   is_duplicate  404287 non-null  int8  
dtypes: int8(1), object(2)
memory usage: 6.6+ MB


In [9]:
def create_train_valid_split(
    df: pd.DataFrame,
    cfg: WordPieceConfig
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    train_df, val_df = train_test_split(
        df,
        test_size=cfg.valid_size,
        random_state=cfg.seed,
        stratify=df["is_duplicate"]
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    print(
        f">>> DataFrame splitted to train and validation...\n"
        f">>> Training rows: {len(train_df):,}\n"
        f">>> Validation rows: {len(val_df):,}\n"
        f">>> Train duplicate rate: "
        f"{train_df['is_duplicate'].mean():.4f}\n"
        f">>> Validation duplicate rate: "
        f"{val_df['is_duplicate'].mean():.4f}"
    )

    return train_df, val_df

In [10]:
def save_splits(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    paths: WordPiecePaths
) -> None:
    train_df.to_csv(paths.train_split_path, index=False)
    val_df.to_csv(paths.valid_split_path, index=False)

    print(
        f">>> Train DataFrame saved to: {paths.train_split_path}\n"
        f">>> Validation DataFrame saved to: {paths.valid_split_path}"
    )

In [11]:
def build_corpus_file(
    train_df: pd.DataFrame,
    paths: WordPiecePaths
) -> None:

    question_count = 0
    with open(paths.corpus_path, "w", encoding="utf-8") as f:
        for row in train_df.itertuples(index=False):
            f.write(f"{row.question1}\n")
            f.write(f"{row.question2}\n")
            question_count += 2

    print(
        f">>> Tokenizer corpus saved to: {paths.corpus_path}"
        f">>> Question counts: {question_count:,}"
    )

In [12]:
train_df, val_df = create_train_valid_split(
    df=df, cfg=cfg
)
save_splits(
    train_df=train_df,
    val_df=val_df,
    paths=paths
)
corpus_question_count = build_corpus_file(
    train_df=train_df,
    paths=paths,
)

>>> DataFrame splitted to train and validation...
>>> Training rows: 363,858
>>> Validation rows: 40,429
>>> Train duplicate rate: 0.3692
>>> Validation duplicate rate: 0.3692
>>> Train DataFrame saved to: data\processed\train_split.csv
>>> Validation DataFrame saved to: data\processed\valid_split.csv
>>> Tokenizer corpus saved to: data\processed\tokenizer_corpus.txt>>> Question counts: 727,716


In [13]:
with paths.corpus_path.open(
    "r",
    encoding="utf-8",
) as file:
    for _ in range(5):
        print(file.readline().strip())

What is maximum steering wheel torque in truck?
What is a steering wheel's torque?
How much does it cost to build a website in India?
How much does it cost to build an ecommerce store in India?
Does time stop ever?


In [14]:
def train_wordpiece_backend(
    paths: WordPiecePaths,
    cfg: WordPieceConfig
) -> Tokenizer:
    tokenizer = Tokenizer(
        WordPiece(
            unk_token=cfg.unk_token,
            continuing_subword_prefix=(
                cfg.continuing_subword_prefix
            ),
            max_input_chars_per_word=(
                cfg.max_input_chars_per_word
            ),
        )
    )

    tokenizer.normalizer = BertNormalizer(
        clean_text=cfg.clean_text,
        handle_chinese_chars=(
            cfg.handle_chinese_chars
        ),
        strip_accents=cfg.strip_accents,
        lowercase=cfg.lowercase
    )

    tokenizer.pre_tokenizer = (
        BertPreTokenizer()
    )

    trainer = WordPieceTrainer(
        vocab_size=cfg.vocab_size,
        min_frequency=cfg.min_frequency,
        show_progress=True,
        special_tokens=cfg.special_tokens,
        continuing_subword_prefix=(
            cfg.continuing_subword_prefix
        )
    )

    tokenizer.train(
        files=[
            str(paths.corpus_path)
        ],
        trainer=trainer
    )

    special_token_ids = {
        token: tokenizer.token_to_id(token)
        for token in cfg.special_tokens
    }

    missing_special_tokens = [
        token for token, token_id in special_token_ids.items()
        if token_id is None
    ]

    if missing_special_tokens:
        raise RuntimeError(
            "The trained vocabulary is missing "
            f"special tokens: {missing_special_tokens}"
        )

    tokenizer.post_processor = (
        TemplateProcessing(
            single=(
                f"{cfg.cls_token} "
                f"$A "
                f"{cfg.sep_token}"
            ),
            pair=(
                f"{cfg.cls_token} "
                f"$A "
                f"{cfg.sep_token} "
                f"$B:1 "
                f"{cfg.sep_token}:1"
            ),
            special_tokens=[
                (
                    cfg.cls_token,
                    special_token_ids[
                        cfg.cls_token
                    ],
                ),
                (
                    cfg.sep_token,
                    special_token_ids[
                        cfg.sep_token
                    ],
                ),
            ],
        )
    )

    tokenizer.decoder = WordPieceDecoder(
        prefix=cfg.continuing_subword_prefix
    )

    print(
        ">>> WordPiece backend trained.\n"
        f">>> Actual vocabulary size: "
        f"{tokenizer.get_vocab_size():,}"
    )

    return tokenizer

In [15]:
backend_tokenizer = train_wordpiece_backend(
    paths=paths,
    cfg=cfg,
)

>>> WordPiece backend trained.
>>> Actual vocabulary size: 30,000


In [16]:
sample_encoding = backend_tokenizer.encode(
    "Tokenization is useful for unseen words."
)
print(sample_encoding.tokens)
print(sample_encoding.ids)

['[CLS]', 'token', '##ization', 'is', 'useful', 'for', 'unseen', 'words', '.', '[SEP]']
[2, 21983, 2610, 1460, 3832, 1484, 27775, 3379, 18, 3]


In [17]:
pair_encoding = backend_tokenizer.encode(
    "How can I learn machine learning?",
    "What is the best way to study machine learning?",
)
print(pair_encoding.tokens)
print(pair_encoding.type_ids)

['[CLS]', 'how', 'can', 'i', 'learn', 'machine', 'learning', '?', '[SEP]', 'what', 'is', 'the', 'best', 'way', 'to', 'study', 'machine', 'learning', '?', '[SEP]']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [18]:
def create_fast_tokenizer(
    backend_tokenizer: Tokenizer,
    cfg: WordPieceConfig
) -> PreTrainedTokenizerFast:

    tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=backend_tokenizer,
        unk_token=cfg.unk_token,
        cls_token=cfg.cls_token,
        sep_token=cfg.sep_token,
        pad_token=cfg.pad_token,
        mask_token=cfg.mask_token,
        model_max_length=cfg.max_pair_length
    )

    tokenizer.padding_side = "right"
    tokenizer.truncation_side = "right"

    if not tokenizer.is_fast:
        raise RuntimeError(
            ">>> Expected a Fast Tokenizer."
        )

    print(
        ">>> Fast tokenizer created.\n"
        f">>> Class: {type(tokenizer).__name__}\n"
        f">>> Vocabulary size: {len(tokenizer):,}"
    )

    return tokenizer

In [19]:
tokenizer = create_fast_tokenizer(
    backend_tokenizer=backend_tokenizer,
    cfg=cfg
)

>>> Fast tokenizer created.
>>> Class: PreTrainedTokenizerFast
>>> Vocabulary size: 30,000


In [20]:
tokenizer.special_tokens_map

{'unk_token': '[UNK]',
 'sep_token': '[SEP]',
 'pad_token': '[PAD]',
 'cls_token': '[CLS]',
 'mask_token': '[MASK]'}

In [ ]:
def save_tokenizer(
    backend_tokenizer: Tokenizer,
    tokenizer: PreTrainedTokenizerFast,
    paths: WordPiecePaths
) -> list[str]:

    paths.tokenizer.mkdir(
        parent=True,
        exist_ok=True
    )

    tokenizer.save_pretrained(
        str(paths.tokenizer_dir)
    )

    backend_tokenizer.model.save(
        str(paths.tokenizer_dir)
    )

    required_paths = [
        paths.tokenizer_json_path,
        paths.tokenizer_config_path,
        paths.vocab_path
    ]

    missing_files = [
        path.name for path in required_paths if not path.is_file()
    ]

    if missing_files:
        raise FileNotFoundError(
            f">>> Missing tokenizer files: {missing_files}"
        )

    saved_files = sorted(
        path.name for path in paths.tokenizer_dir.iterdir() if path.is_file()
    )

    print(">>> Saved tokenizer files:")

    for filename in saved_files:
        print(f"    - {filename}")

    if not paths.special_tokens_map_path.exists():
        print(
            ">>> special_tokens_map.json was not "
            "created by this Transformers version. "
            "This is acceptable if reload validation passes."
        )

    return saved_files